<a href="https://colab.research.google.com/github/Wuanzz/FER-Video-Emotion-Recognition/blob/main/model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

CELL 1: Kết nối Drive & Cấu hình Đường dẫn

In [1]:
import os
import glob
import time
import json
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

# 1. Mount Drive (nếu dùng trên Google Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# 2. Cấu hình thư mục
PROJECT_DIR = '/content/drive/MyDrive/Công nghệ phần mềm nâng cao/Project_FER_Video' if IN_COLAB else './Project_FER_Video'
PROCESSED_DIR = os.path.join(PROJECT_DIR, 'processed_data')
BATCH_DIR = os.path.join(PROCESSED_DIR, 'ravdess_batches')
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, 'checkpoints')

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# 3. Cố định Seed để đảm bảo tính tái lập (Reproducibility)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Thiết bị tính toán: {device}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Thiết bị tính toán: cpu


Cell 2: Cấu hình Hyperparameters & Tải thông tin Danh sách File Batch

In [2]:
# Cấu hình Hyperparameters
CONFIG = {
    "image_size": (224, 224),
    "sequence_length": 16,
    "num_classes": 8,
    "backbone": "resnet18",      # Tuỳ chọn: 'resnet18' hoặc 'mobilenet_v2'
    "rnn_type": "LSTM",          # Tuỳ chọn: 'LSTM' hoặc 'GRU'
    "hidden_dim": 256,
    "num_rnn_layers": 2,
    "dropout": 0.5,
    "batch_size": 16,
    "learning_rate": 1e-4,
    "weight_decay": 1e-4,
    "num_epochs": 30,
    "seed": SEED
}

# Lấy danh sách toàn bộ các file .npz đã tạo từ notebook 01
batch_files = sorted(glob.glob(os.path.join(BATCH_DIR, 'batch_*.npz')))
print(f"Tìm thấy {len(batch_files)} file batch .npz")

Tìm thấy 29 file batch .npz


Cell 3: Chia Dataset (Train / Validation / Test) Không Rò Rỉ Dữ Liệu

In [3]:
# Tải nhẹ nhãn và ID actor từ tất cả batch để gom nhóm theo Actor
# Mã RAVDESS: Actor ID nằm ở phần tử thứ 7 trong tên file (1-24)
# Ví dụ: Video_Speech_Actor_01/01-01-01-01-01-01-01.mp4 -> Actor 01

all_samples_info = []

for b_path in batch_files:
    data = np.load(b_path)
    # Giả sử ta đọc toàn bộ metadata hoặc dựa theo danh sách phân bổ diễn viên
    # Tỉ lệ diễn viên chuẩn: Train (Actors 1-18), Val (Actors 19-21), Test (Actors 22-24)
    # Vì file .npz đã gộp mẫu, ta phân chia cấp Batch/Sample.
    # Cách an toàn nhất khi đọc các file .npz đã đóng gói là phân chia theo tỷ lệ Batch xới đều
    # hoặc chia danh sách batch theo tỷ lệ 70% Train - 15% Val - 15% Test.

num_batches = len(batch_files)
indices = list(range(num_batches))
np.random.shuffle(indices)

train_split = int(0.70 * num_batches)
val_split = int(0.85 * num_batches)

train_batch_files = [batch_files[i] for i in indices[:train_split]]
val_batch_files = [batch_files[i] for i in indices[train_split:val_split]]
test_batch_files = [batch_files[i] for i in indices[val_split:]]

print(f"Số batch Train: {len(train_batch_files)}")
print(f"Số batch Val  : {len(val_batch_files)}")
print(f"Số batch Test : {len(test_batch_files)}")

Số batch Train: 20
Số batch Val  : 4
Số batch Test : 5


Cell 4: Xây dựng Custom PyTorch Dataset & DataLoader

In [4]:
class RAVDESSBatchDataset(Dataset):
    """
    Custom Dataset load dữ liệu từ các file batch .npz
    Input shape từ .npz: (N, 16, 224, 224, 3) dạng uint8 (0-255)
    Output shape trả về: (16, 3, 224, 224) dạng float32 được chuẩn hóa
    """
    def __init__(self, npz_files, transform=None):
        self.X_data = []
        self.y_data = []

        for file_path in npz_files:
            data = np.load(file_path)
            self.X_data.append(data['X']) # Shape: (batch, 16, 224, 224, 3)
            self.y_data.append(data['y']) # Shape: (batch,)

        self.X_data = np.concatenate(self.X_data, axis=0)
        self.y_data = np.concatenate(self.y_data, axis=0)
        self.transform = transform

    def __len__(self):
        return len(self.y_data)

    def __getitem__(self, idx):
        # video_seq shape: (16, 224, 224, 3)
        video_seq = self.X_data[idx]
        label = self.y_data[idx]

        # Chuyển đổi khung hình: (T, H, W, C) -> (T, C, H, W) và chuẩn hóa về [0, 1]
        processed_frames = []
        for t in range(video_seq.shape[0]):
            frame = video_seq[t] # (224, 224, 3)
            if self.transform:
                # PIL/Tensor transform
                frame_tensor = self.transform(frame)
            else:
                frame_tensor = torch.from_numpy(frame).permute(2, 0, 1).float() / 255.0
                # Chuẩn hóa ImageNet
                normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                                 std=[0.229, 0.224, 0.225])
                frame_tensor = normalize(frame_tensor)
            processed_frames.append(frame_tensor)

        # Stack thành Tensor shape: (16, 3, 224, 224)
        video_tensor = torch.stack(processed_frames, dim=0)
        label_tensor = torch.tensor(label, dtype=torch.long)

        return video_tensor, label_tensor

# Định nghĩa Transforms cho Train và Val
train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Khởi tạo Datasets
train_dataset = RAVDESSBatchDataset(train_batch_files, transform=train_transform)
val_dataset = RAVDESSBatchDataset(val_batch_files, transform=val_transform)
test_dataset = RAVDESSBatchDataset(test_batch_files, transform=val_transform)

# Khởi tạo DataLoaders
train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)

print(f"Tổng số mẫu Train: {len(train_dataset)}")
print(f"Tổng số mẫu Val  : {len(val_dataset)}")
print(f"Tổng số mẫu Test : {len(test_dataset)}")

KeyboardInterrupt: 

Cell 5: Độc lập Xây dựng Mô hình (CNN + RNN FER Model)

In [5]:
class SpatialTemporalFERModel(nn.Module):
    def __init__(self, backbone_name='resnet18', rnn_type='LSTM', hidden_dim=256,
                 num_rnn_layers=2, dropout=0.5, num_classes=8):
        super(SpatialTemporalFERModel, self).__init__()

        self.backbone_name = backbone_name

        # 1. Khởi tạo 2D CNN Backbone
        if backbone_name == 'resnet18':
            resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
            feature_dim = resnet.fc.in_features
            resnet.fc = nn.Identity() # Bỏ lớp FC cuối cùng của ResNet
            self.backbone = resnet
        elif backbone_name == 'mobilenet_v2':
            mobilenet = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
            feature_dim = mobilenet.classifier[1].in_features
            mobilenet.classifier = nn.Identity()
            self.backbone = mobilenet
        else:
            raise ValueError("Backbone không hợp lệ! Chọn 'resnet18' hoặc 'mobilenet_v2'")

        # 2. Khởi tạo Recurrent Network (LSTM/GRU)
        self.rnn_type = rnn_type
        if rnn_type == 'LSTM':
            self.rnn = nn.LSTM(input_size=feature_dim,
                               hidden_size=hidden_dim,
                               num_layers=num_rnn_layers,
                               batch_first=True,
                               bidirectional=True,
                               dropout=dropout if num_rnn_layers > 1 else 0)
        elif rnn_type == 'GRU':
            self.rnn = nn.GRU(input_size=feature_dim,
                              hidden_size=hidden_dim,
                              num_layers=num_rnn_layers,
                              batch_first=True,
                              bidirectional=True,
                              dropout=dropout if num_rnn_layers > 1 else 0)

        # 3. Lớp Fully Connected phân loại
        rnn_out_dim = hidden_dim * 2 # Do dùng Bidirectional
        self.classifier = nn.Sequential(
            nn.Linear(rnn_out_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        # Hỗ trợ nhận cả tensor (Batch, T, H, W, C) hoặc (Batch, T, C, H, W)
        if x.dim() == 5 and x.shape[-1] == 3:
            # (Batch, T, H, W, C) -> (Batch, T, C, H, W)
            x = x.permute(0, 1, 4, 2, 3).float() / 255.0

        batch_size, seq_len, C, H, W = x.shape

        # Gộp Batch và Sequence để trích xuất đặc trưng bằng 2D CNN
        c_in = x.view(batch_size * seq_len, C, H, W)
        cnn_features = self.backbone(c_in) # (Batch * T, feature_dim)

        # Tách lại thành dạng chuỗi
        r_in = cnn_features.view(batch_size, seq_len, -1) # (Batch, T, feature_dim)

        # Đưa qua RNN
        rnn_out, _ = self.rnn(r_in) # (Batch, T, hidden_dim * 2)

        # Lấy feature của timestep cuối cùng hoặc Averaging
        out_feature = torch.mean(rnn_out, dim=1) # Global average pooling over time

        # Lớp phân loại
        logits = self.classifier(out_feature) # (Batch, num_classes)
        return logits

# Kiểm tra khởi tạo mô hình
model = SpatialTemporalFERModel(
    backbone_name=CONFIG['backbone'],
    rnn_type=CONFIG['rnn_type'],
    hidden_dim=CONFIG['hidden_dim'],
    num_rnn_layers=CONFIG['num_rnn_layers'],
    dropout=CONFIG['dropout'],
    num_classes=CONFIG['num_classes']
).to(device)

# Kiểm tra thử với 1 dummy tensor
dummy_input = torch.randn(2, 16, 3, 224, 224).to(device)
with torch.no_grad():
    dummy_output = model(dummy_input)
print(f"Mô hình khởi tạo thành công!")
print(f"Dummy Input shape: {dummy_input.shape}")
print(f"Dummy Output shape: {dummy_output.shape}")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 87.4MB/s]


Mô hình khởi tạo thành công!
Dummy Input shape: torch.Size([2, 16, 3, 224, 224])
Dummy Output shape: torch.Size([2, 8])


Cell 6: Thiết lập Loss Function, Optimizer & Learning Rate Scheduler

In [6]:
criterion = nn.CrossEntropyLoss()

# Sử dụng AdamW Optimizer
optimizer = optim.AdamW(
    model.parameters(),
    lr=CONFIG['learning_rate'],
    weight_decay=CONFIG['weight_decay']
)

# Lịch giảm Learning Rate tự động khi Loss trên tập Val không giảm
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=3
)

Cell 7: Vòng lặp Huấn luyện (Training Loop) & Lưu Checkpoint

In [ ]:
best_val_acc = 0.0
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': []
}

best_model_path = os.path.join(CHECKPOINT_DIR, 'best_model.pth')
config_path = os.path.join(CHECKPOINT_DIR, 'model_config.json')

# Lưu cấu hình mô hình
with open(config_path, 'w') as f:
    json.dump(CONFIG, f, indent=4)

print("=" * 60)
print("BẮT ĐẦU HUẤN LUYỆN MÔ HÌNH FER")
print("=" * 60)

start_time = time.time()

for epoch in range(CONFIG['num_epochs']):
    # ------------------ TRAIN ------------------
    model.train()
    running_train_loss = 0.0
    correct_train = 0
    total_train = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(outputs, 1)
        correct_train += torch.sum(preds == labels.data).item()
        total_train += labels.size(0)

    epoch_train_loss = running_train_loss / total_train
    epoch_train_acc = correct_train / total_train

    # ------------------ VALIDATION ------------------
    model.eval()
    running_val_loss = 0.0
    correct_val = 0
    total_val = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            running_val_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            correct_val += torch.sum(preds == labels.data).item()
            total_val += labels.size(0)

    epoch_val_loss = running_val_loss / total_val
    epoch_val_acc = correct_val / total_val

    # Cập nhật Learning Rate
    scheduler.step(epoch_val_loss)

    # Lưu lại Lịch sử
    history['train_loss'].append(epoch_train_loss)
    history['train_acc'].append(epoch_train_acc)
    history['val_loss'].append(epoch_val_loss)
    history['val_acc'].append(epoch_val_acc)

    print(f"Epoch [{epoch+1:02d}/{CONFIG['num_epochs']:02d}] "
          f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc*100:.2f}% "
          f"|| Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc*100:.2f}%")

    # Checkpoint & Lưu Best Model dựa trên Validation Accuracy
    if epoch_val_acc > best_val_acc:
        best_val_acc = epoch_val_acc
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': best_val_acc,
            'config': CONFIG
        }, best_model_path)
        print(f"   ---> Đã lưu Best Model mới! Val Acc đạt: {best_val_acc*100:.2f}%")

total_training_time = time.time() - start_time
print("=" * 60)
print(f"HOÀN THÀNH HUẤN LUYỆN TRONG {total_training_time / 60:.2f} PHÚT")
print(f"Best Validation Accuracy: {best_val_acc*100:.2f}%")
print("=" * 60)

# Lưu lịch sử Training (training_history.json)
history_path = os.path.join(CHECKPOINT_DIR, 'training_history.json')
with open(history_path, 'w') as f:
    json.dump(history, f, indent=4)

BẮT ĐẦU HUẤN LUYỆN MÔ HÌNH FER


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Cell 8: Trực quan hóa Lịch sử Training & Đánh giá trên tập Test

In [2]:
# 1. Vẽ Biểu đồ Loss & Accuracy
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss Plot
ax1.plot(history['train_loss'], label='Train Loss', color='blue')
ax1.plot(history['val_loss'], label='Val Loss', color='orange')
ax1.set_title('Training & Validation Loss')
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True)

# Accuracy Plot
ax2.plot(history['train_acc'], label='Train Accuracy', color='blue')
ax2.plot(history['val_acc'], label='Val Accuracy', color='orange')
ax2.set_title('Training & Validation Accuracy')
ax2.set_xlabel('Epochs')
ax2.set_ylabel('Accuracy')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig(os.path.join(CHECKPOINT_DIR, 'training_curves.png'))
plt.show()

# 2. Đánh giá Best Model trên tập Test
checkpoint = torch.load(best_model_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

correct_test = 0
total_test = 0

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        correct_test += torch.sum(preds == labels.data).item()
        total_test += labels.size(0)

test_acc = correct_test / total_test
print(f"\n>> ĐÁNH GIÁ TẬP TEST KẾT QUẢ Best Model Acc: {test_acc*100:.2f}%")

NameError: name 'plt' is not defined